<div align="center">

<img src="https://s3.amazonaws.com/files.pucp.edu.pe/pucp-general/img-header/logo-pucp-new.svg" alt="Pontificia Universidad Católica del Perú">

<br><br>

## Pontificia Universidad Católica del Perú  

### Deep Learning con Python  

<br>

## Clase 2 — Laboratorio 2  

### Construyendo una red neuronal

### 🐶 Dogs vs Non-Dogs  

</div>

---

📥 Red neuronal dogs vs non_dogs

Objetivo de esta celda:

- Cargar el dataset de dogs vs non_dogs

- Desarrollar la arquitectura de la red neuronal

- Entrenar el modelo y validar sus metricas

In [1]:
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import copy
import matplotlib.pyplot as plt

🧠 Carga y Preparación del Dataset

En esta sección se define una función personalizada para cargar el dataset de imágenes desde la estructura de carpetas previamente construida.

La función se encarga de leer las imágenes, redimensionarlas, convertirlas a arreglos numéricos y construir las matrices de datos (X) y etiquetas (y) necesarias para el entrenamiento del modelo.

Posteriormente, el dataset es dividido en conjuntos de entrenamiento y prueba, permitiendo evaluar el desempeño del modelo en datos no vistos.

Finalmente, las etiquetas son reestructuradas para mantener un formato adecuado para el pipeline de entrenamiento.

Objetivo de esta celda:

- Cargar imágenes desde las carpetas del dataset

- Convertir imágenes a arreglos numéricos

- Construir las matrices de datos y etiquetas

- Dividir el dataset en train y test

- Preparar los datos para el modelo

In [ ]:
def load_dataset_custom(path="dataset_v1_resized", img_size=(128, 128)):

    X = [] # Es una lista
    y = [] # Es una lista

    classes = ["non_dog", "dog"]

    for label, class_name in enumerate(classes): #etiqueta y nombre de la clase
        class_path = os.path.join(path, class_name) #ruta de la carpeta de la clase

        for file in os.listdir(class_path): #lista de archivos en la carpeta de la clase
            img_path = os.path.join(class_path, file) #ruta de la imagen

            try:
                img = Image.open(img_path).convert("RGB") # Lee la imagen y garantiza 3 canales (R, G, B)
                img = img.resize(img_size) # Estandariza a 128x128 → todas las imgs del mismo tamaño
                img_array = np.array(img)
                # Convierte a numpy: shape = (128, 128, 3)
#                                              ↑    ↑   ↑
#                                             alto ancho RGB

                X.append(img_array) # Es una lista de numpy
                y.append(label) # Es una lista de enteros

            except:
                print(f"Error leyendo {img_path}")

    X = np.array(X) #convierte la lista X a un array numpy
    y = np.array(y) #convierte la lista y a un array numpy

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    y_train = y_train.reshape(1, y_train.shape[0])
    y_test = y_test.reshape(1, y_test.shape[0])

    return X_train, y_train, X_test, y_test, np.array(classes)

### Visualización de `img_array` para una imagen:
```
img_array[0][0] = [123, 45, 200]   ← pixel (0,0): R=123, G=45, B=200
img_array[0][1] = [100, 80, 190]   ← pixel (0,1)
...
img_array[127][127] = [90, 60, 150] ← último pixel

In [4]:
train_set_x_orig, train_set_y, test_set_x_orig, test_set_y, classes = load_dataset_custom()

In [5]:
train_set_x_orig.shape

(240, 128, 128, 3)

🖼️ Visualización de Ejemplos del Conjunto de Entrenamiento

En esta sección se visualiza una imagen del conjunto de entrenamiento junto con su etiqueta correspondiente para verificar que el proceso de carga del dataset se realizó correctamente.

Esta inspección permite confirmar que las imágenes se representan adecuadamente y que las etiquetas coinciden con la clase real, asegurando la correcta preparación de los datos antes de continuar con el proceso de modelado.

In [ ]:
index = 67
plt.imshow(train_set_x_orig[index])
plt.show()
label = int(np.squeeze(train_set_y[:, index]))
print(f"y = {label}, it's a '{classes[label]}' picture.")

📊 Inspección de Dimensiones del Dataset

En esta sección se analizan las dimensiones del conjunto de datos para comprender su estructura antes de iniciar el entrenamiento del modelo.

Se calcula:

- El número de ejemplos de entrenamiento

- El número de ejemplos de prueba

- Las dimensiones espaciales de cada imagen

- La forma completa de cada muestra (alto, ancho y canales RGB)

Esta verificación permite asegurar que los datos tienen el formato esperado y que el modelo recibirá entradas con dimensiones consistentes.

In [ ]:
m_train = train_set_x_orig.shape[0]
m_test = test_set_x_orig.shape[0]
num_px = train_set_x_orig.shape[1]

print("Number of training examples:", m_train)
print("Number of testing examples:", m_test)
print("Height/Width of each image:", num_px)
print("Each image is of size:", (num_px, num_px, 3))

🔄 Reestructuración y Normalización de los Datos

En esta sección se transforman las imágenes en un formato adecuado para el entrenamiento del modelo.

Primero, cada imagen es aplanada (flattened), convirtiendo su estructura tridimensional (alto × ancho × canales) en un vector columna. Este paso es necesario cuando se trabaja con modelos que reciben vectores de características como entrada.

Posteriormente, se realiza la normalización de los datos, dividiendo los valores de los píxeles entre 255. Esto permite escalar los valores al rango 
[
0
,
1
]
[0,1], mejorando la estabilidad numérica y facilitando la convergencia durante el entrenamiento.

Objetivo de esta etapa:

- Convertir imágenes en vectores de entrada

- Asegurar compatibilidad con el modelo

- Escalar los datos para optimizar el aprendizaje

In [ ]:
train_set_x_flatten = train_set_x_orig.reshape(train_set_x_orig.shape[0], -1).T
test_set_x_flatten = test_set_x_orig.reshape(test_set_x_orig.shape[0], -1).T

print("train_set_x_flatten shape:", train_set_x_flatten.shape)
print("train_set_y shape:", train_set_y.shape)
print("test_set_x_flatten shape:", test_set_x_flatten.shape)
print("test_set_y shape:", test_set_y.shape)

train_set_x = train_set_x_flatten / 255.
test_set_x = test_set_x_flatten / 255.

👇

🔢 Función de Activación: Sigmoid

En esta sección se implementa la función sigmoid, una función de activación ampliamente utilizada en problemas de clasificación binaria.

La función sigmoid transforma un valor real 
𝑧
z en un valor dentro del rango 
(
0
,
1
)
(0,1), lo que permite interpretarlo como una probabilidad.

Matemáticamente se define como:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$


Esta función será utilizada en la etapa de forward propagation para convertir la combinación lineal de entradas en una probabilidad de pertenencia a la clase positiva (dog).

Importancia:

- Permite modelar problemas de clasificación binaria

- Introduce no linealidad al modelo

- Produce salidas interpretables como probabilidades

In [ ]:
def sigmoid(z):
    return None

⚙️ Inicialización de Parámetros

En esta sección se definen los parámetros iniciales del modelo: el vector de pesos w y el sesgo b.

Los pesos se inicializan como un vector de ceros con dimensión igual al número de características de entrada, mientras que el sesgo se inicializa en 0.

Esta inicialización es el punto de partida del proceso de entrenamiento, donde posteriormente los parámetros serán ajustados mediante el algoritmo de optimización (Gradient Descent).

¿Qué representan estos parámetros?

- w → pesos asociados a cada característica de entrada

- b → término de sesgo (bias)

- Juntos modelan la combinación lineal:

$$
z = w^T x + b
$$

In [ ]:
def initialize_with_zeros(dim):
    w = None
    b = None
    return w, b

## 🔄 Forward y Backward Propagation

En esta sección se implementa el proceso completo de propagación para la regresión logística. Este procedimiento permite calcular las predicciones del modelo, evaluar el error y obtener los gradientes necesarios para actualizar los parámetros.

---

### 🔹 Forward Propagation

Primero se calcula la combinación lineal:

$$
z = w^T X + b
$$

Luego se aplica la función sigmoid para obtener las activaciones:

$$
A = \sigma(z) = \frac{1}{1 + e^{-z}}
$$

Posteriormente, se calcula la función de costo (Negative Log-Likelihood):

$$
J(w,b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ Y^{(i)} \log(A^{(i)}) + (1 - Y^{(i)}) \log(1 - A^{(i)}) \right]
$$

---

### 🔹 Backward Propagation

Se calculan los gradientes respecto a los parámetros:

$$
dw = \frac{1}{m} X (A - Y)^T
$$

$$
db = \frac{1}{m} \sum (A - Y)
$$

Estos gradientes permiten actualizar los parámetros mediante Gradient Descent en la siguiente etapa del entrenamiento.

---

### 🎯 Objetivo de esta función

- Calcular las predicciones del modelo  
- Evaluar el costo  
- Obtener los gradientes para optimización  

In [ ]:
def propagate(w, b, X, Y):

    z=np.dot(w.T,X)+b
    A=sigmoid(z)

    m=X.shape[1]

    cost = -1/m * np.sum(Y*np.log(A)+(1-Y)*np.log(1-A))

    dw = 1/m * np.dot(X,(A-Y).T)
    db=1/m * np.sum(A-Y)

    grads = {"dw": dw,
             "db": db}
    
    return grads, cost

## 📉 Optimización mediante Gradient Descent

En esta sección se implementa el algoritmo de **Gradient Descent**, el cual permite optimizar los parámetros del modelo (w y b) minimizando la función de costo.

El procedimiento iterativo consiste en dos pasos fundamentales:

---

### 🔹 1. Cálculo del costo y gradientes

En cada iteración se ejecuta la función `propagate()` para obtener:

- El costo actual del modelo
- Los gradientes respecto a los parámetros

$$
dw = \frac{\partial J}{\partial w}
$$

$$
db = \frac{\partial J}{\partial b}
$$

---

### 🔹 2. Actualización de parámetros

Los parámetros se actualizan utilizando la regla de Gradient Descent:

$$
w := w - \alpha \, dw
$$

$$
b := b - \alpha \, db
$$

donde:

- $\alpha$ es el *learning rate*
- $dw$ y $db$ son los gradientes calculados
- $J(w,b)$ es la función de costo

---

### 🔁 Proceso Iterativo

Este procedimiento se repite durante un número determinado de iteraciones:

$$
\text{Repeat for } i = 1,2,...,n
$$

En cada paso, el modelo ajusta sus parámetros para reducir progresivamente el error.

---

### 🎯 Objetivo de esta función

- Minimizar la función de costo  
- Encontrar los valores óptimos de $w$ y $b$  
- Registrar la evolución del costo para analizar la convergencia  


In [ ]:
def optimize(w, b, X, Y, num_iterations=100, learning_rate=0.009, print_cost=False):
    params = None
    grads = None
    costs = None

    for i in range(num_iterations):
        grads, cost = propagate(w, b, X, Y)
        dw=grads["dw"]
        db=grads["db"]

        w=w-learning_rate*dw
        b=b-learning_rate*db

        if i % 100 == 0:
            costs.append(cost)

        if print_cost and i % 100 == 0:
            print(f"Costo tras {i} iteraciones: {cost}")

    params = {"w": w,
              "b": b}
        
    return params, grads, costs

## 🎯 Función de Predicción

En esta sección se implementa la función `predict`, la cual utiliza los parámetros aprendidos del modelo ($w$ y $b$) para generar predicciones sobre nuevos datos.

---

### 🔹 Cálculo de probabilidades

Primero se calcula la combinación lineal:

$$
z = w^T X + b
$$

Luego se aplica la función sigmoid para obtener las probabilidades:

$$
A = \sigma(z) = \frac{1}{1 + e^{-z}}
$$

Cada valor en $A$ representa la probabilidad de que la imagen pertenezca a la clase positiva (dog).

---

### 🔹 Conversión a etiquetas binarias

Las probabilidades se transforman en predicciones finales usando un umbral de decisión:

$$
\hat{y} =
\begin{cases}
1 & \text{si } A > 0.5 \\
0 & \text{si } A \le 0.5
\end{cases}
$$

Donde:

- 1 → dog  
- 0 → non_dog  

---

### 🎯 Objetivo de esta función

- Generar predicciones finales del modelo  
- Convertir probabilidades en clases binarias  
- Evaluar el desempeño del modelo en datos nuevos  


In [ ]:
def predict(w, b, X):
    Y_prediction = None
    
    return Y_prediction

## 🧠 Construcción del Modelo Completo

En esta sección se implementa la función `model`, la cual integra todas las etapas desarrolladas previamente para construir y entrenar un modelo de **Regresión Logística** desde cero.

Esta función ejecuta el flujo completo de aprendizaje supervisado:

---

### 🔹 1. Inicialización de parámetros

Se inicializan los parámetros del modelo:

$$
w = 0, \quad b = 0
$$

donde:

- $w$ representa los pesos del modelo  
- $b$ corresponde al término de sesgo (*bias*)

---

### 🔹 2. Optimización mediante Gradient Descent

Se ejecuta el algoritmo de optimización para minimizar la función de costo:

$$
(w, b) := \text{GradientDescent}(w, b)
$$

Durante este proceso, los parámetros se actualizan iterativamente utilizando los gradientes calculados en la etapa de propagación.

---

### 🔹 3. Generación de predicciones

Una vez entrenado el modelo, se generan predicciones tanto para el conjunto de entrenamiento como para el conjunto de prueba:

$$
\hat{Y} = \text{predict}(w, b, X)
$$

---

### 🔹 4. Evaluación del modelo

El desempeño del modelo se evalúa mediante la métrica de **accuracy**:

$$
\text{Accuracy} = 100 - \frac{1}{m}\sum |\hat{Y} - Y| \times 100
$$

Esto permite medir qué tan bien el modelo clasifica imágenes no vistas previamente.

---

### 🎯 Objetivo de esta función

- Integrar todas las etapas del modelo  
- Entrenar la regresión logística  
- Generar predicciones  
- Evaluar el desempeño en train y test  
- Almacenar información relevante del entrenamiento  


In [ ]:
def model(X_train, Y_train, X_test, Y_test, num_iterations=2000, learning_rate=0.5, print_cost=False):
    d = None
    return d

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=2000, learning_rate=0.005, print_cost=True)

## 📈 Curva de Aprendizaje

En esta sección se visualiza la evolución de la función de costo durante el entrenamiento del modelo.

La gráfica muestra cómo el costo disminuye a medida que avanzan las iteraciones del algoritmo de Gradient Descent.

---

### 🔹 Interpretación

- El eje horizontal representa las iteraciones (cada punto corresponde a 100 iteraciones).
- El eje vertical representa el valor de la función de costo.
- Una curva descendente indica que el modelo está aprendiendo correctamente.
- Si el costo no disminuye o diverge, puede ser necesario ajustar el *learning rate*.

---

### 🔎 Análisis

Si el modelo está correctamente implementado, deberíamos observar:

$$
J(w,b) \downarrow \text{ a medida que aumentan las iteraciones}
$$

Esto confirma que el algoritmo de optimización está convergiendo hacia una solución óptima.

---

### 🎯 Objetivo de esta visualización

- Evaluar la convergencia del modelo  
- Analizar el comportamiento del learning rate  
- Detectar posibles problemas de entrenamiento  


## 📊 Comparación de Learning Rates

En esta sección se entrenan múltiples modelos utilizando diferentes valores de *learning rate* con el objetivo de analizar su impacto en la convergencia del algoritmo de Gradient Descent.

El *learning rate* ($\alpha$) controla el tamaño del paso en cada actualización de los parámetros:

$$
w := w - \alpha \, dw
$$

$$
b := b - \alpha \, db
$$

---

### 🔎 Objetivo del experimento

Se entrenan varios modelos con distintos valores de $\alpha$ y se comparan sus curvas de costo:

- $\alpha = 0.001$
- $\alpha = 0.0005$
- $\alpha = 0.0001$

Luego se grafican sus respectivas curvas de aprendizaje para observar:

- Velocidad de convergencia  
- Estabilidad del entrenamiento  
- Comportamiento del costo  

---

### 📈 Interpretación esperada

- Un **learning rate muy grande** puede generar oscilaciones o divergencia.
- Un **learning rate muy pequeño** puede hacer que el entrenamiento sea demasiado lento.
- El valor óptimo permite una disminución suave y estable del costo.

---

### 🎯 Propósito

Este análisis permite seleccionar un valor adecuado de learning rate que garantice:

- Convergencia estable  
- Entrenamiento eficiente  
- Mejor desempeño del modelo  


## 🖼️ Preprocesamiento de una Imagen Externa para Predicción

En esta sección se prepara una imagen externa para que pueda ser utilizada como entrada del modelo entrenado.

Dado que el modelo fue entrenado con imágenes de tamaño **128 × 128 píxeles**, es necesario aplicar el mismo preprocesamiento a cualquier nueva imagen antes de realizar una predicción.

---

### 🔹 1. Recorte centrado (Center Crop)

Primero se convierte la imagen en un formato cuadrado recortando la región central:

- Se identifica la dimensión mínima (ancho o alto).
- Se recorta la imagen manteniendo el centro.
- Esto evita deformaciones al redimensionar.

---

### 🔹 2. Redimensionamiento

Luego la imagen se redimensiona a:

$$
128 \times 128
$$

Utilizando interpolación de alta calidad (`LANCZOS`).

---

### 🎯 Objetivo

- Asegurar consistencia con el dataset de entrenamiento  
- Mantener proporciones correctas  
- Preparar la imagen para el pipeline de predicción  

Este paso es fundamental para que el modelo reciba datos en el mismo formato con el que fue entrenado.


In [ ]:
img = Image.open("image_ex6.png")
width, height = img.size
min_dim = min(width, height)
left = (width - min_dim) // 2
top = (height - min_dim) // 2
right = (width + min_dim) // 2
bottom = (height + min_dim) // 2
img_cropped = img.crop((left, top, right, bottom))
img_resized = img_cropped.resize((128, 128), Image.LANCZOS)
img_resized.save("perro_128x128.jpg")
print("Imagen cuadrada y convertida a 128x128 😼")

## 🔮 Predicción sobre una Imagen Externa

En esta sección se realiza una predicción utilizando una imagen nueva que no formó parte del conjunto de entrenamiento.

Para garantizar compatibilidad con el modelo, se aplican los mismos pasos de preprocesamiento utilizados durante el entrenamiento:

---

### 🔹 1. Carga y visualización

Se carga la imagen y se muestra para verificar que fue leída correctamente.

---

### 🔹 2. Normalización

Los valores de los píxeles se escalan al rango:

$$
[0,1]
$$

dividiendo entre 255, tal como se hizo con el dataset de entrenamiento.

---

### 🔹 3. Vectorización (Flattening)

La imagen se transforma en un vector columna de dimensión:

$$
(128 \times 128 \times 3, \; 1)
$$

para que pueda ser procesada por el modelo.

---

### 🔹 4. Predicción

Se utiliza la función:

$$
\hat{Y} = \text{predict}(w, b, X)
$$

para obtener la clase predicha.

---

### 🎯 Resultado

El modelo clasifica la imagen como:

- **1 → dog**
- **0 → non_dog**

Este paso demuestra el uso práctico del modelo entrenado sobre datos reales no vistos previamente.


In [ ]:
fname = "perro_128x128.jpg"   
image = np.array(Image.open(fname).resize((num_px, num_px)))
plt.imshow(image)
image = image / 255.
image = image.reshape((1, num_px * num_px * 3)).T
my_predicted_image = predict(logistic_regression_model["w"], logistic_regression_model["b"], image)
print("y = " + str(np.squeeze(my_predicted_image)) + ", your algorithm predicts a \"" + classes[int(np.squeeze(my_predicted_image)),] +  "\" picture.")